# Lab 3（上）五種模型，看它們怎麼畫線

同一份資料，五種模型各跑一次，**每種都把它的判斷畫成一張圖**。

KNN → 決策樹 → 隨機森林 → XGBoost → KMeans

最後**把資料切成兩份**——一份給模型念、一份藏起來當考卷，看誰是真的學會、誰只是把答案背下來。

In [ ]:
# 📦 先跑這一格：指定版本，避免學校電腦裝到不相容的舊版（裝不起來看 README）
!pip install -q scikit-learn==1.6.1 pandas==2.2.3 numpy==1.26.4 matplotlib==3.10.0 xgboost==3.2.0

## 🔧 第 0 步：環境檢查

四個版本號都印出來就可以開始。

In [ ]:
# 老師的小設定：載入今天整本要用的工具、設好中文字型（直接跑、不用改）
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# 讓圖上的中文正常顯示（清單第一個是本機認得的字型，其餘是不同系統的備援）
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP", "Noto Sans CJK TC", "Microsoft JhengHei", "PingFang TC", "AR PL UMing CN", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

import sklearn
import xgboost
from sklearn.model_selection import train_test_split

print("sklearn   ", sklearn.__version__)
print("pandas    ", pd.__version__)
print("matplotlib", matplotlib.__version__)
print("xgboost   ", xgboost.__version__)
print("環境 OK，可以開始")

---
## 🟦 A・今天的資料：100 家公司

每家只記兩個數字：**毛利率**（本業賺不賺）、**負債比**（借多少）。答案是漲（1）或跌（0）。

只用兩個數字，是為了能畫成一張平面圖——這樣「模型把平面切成哪幾塊」看得見。

### A1・把 100 家公司生出來

這格直接跑。**註解裡寫了我們是照什麼規則編的。**

In [ ]:
# 這一格直接跑就好，重點在註解寫的「規則」
rng = np.random.default_rng(42)          # 固定亂數種子 → 每個人生出來的 100 家完全一樣

# 刻意編出兩種公司，各 50 家：
#   族群一「穩健傳產型」：毛利率中等、負債比低   → 圖的左下角
#   族群二「擴張科技型」：毛利率高、負債比也高   → 圖的右上角
m1 = rng.normal(30, 10, 50)
d1 = rng.normal(33, 10, 50)
m2 = rng.normal(55, 10, 50)
d2 = rng.normal(58, 10, 50)

margin = np.concatenate([m1, m2]).clip(5, 80).round(1)
debt = np.concatenate([d1, d2]).clip(5, 90).round(1)

# 🔑 漲跌的「真規則」：毛利率減掉負債比，還有剩就判漲
#    白話：本業賺的錢要夠還債，才看好它
label = ((margin - debt) > -3).astype(int)

# 但市場不會這麼乖 —— 挑 10 家把答案翻過來（好公司也會跌、爛公司也會漲）
flip = rng.choice(100, size=10, replace=False)
label[flip] = 1 - label[flip]

companies = pd.DataFrame({"margin": margin, "debt": debt, "label": label})
print("一共幾家：", len(companies))
print("漲（1）幾家：", int(companies["label"].sum()))
print("跌（0）幾家：", int((1 - companies["label"]).sum()))
companies.head()

### A2・先畫出來看一眼

拿到新資料的第一件事。

In [ ]:
cmap_bg = ListedColormap(["#ffd9d9", "#d9e8ff"])   # 底色：淡紅＝判跌、淡藍＝判漲
cmap_pt = ListedColormap(["#d62728", "#1f77b4"])   # 圓點：紅＝真的跌、藍＝真的漲

X = companies[["margin", "debt"]].values     # 特徵表：每家兩個數字
y = companies["label"].values                # 答案：每家一個 0/1

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap_pt, edgecolors="k", s=45)
ax.set_xlabel("毛利率 (%)")
ax.set_ylabel("負債比 (%)")
ax.set_title("100 家公司（藍＝漲、紅＝跌）")
plt.show()

**🔑 三件事：** ① 點分成左下、右上**兩坨**（講 KMeans 時會回來找）；② 分界大致是一條**斜線**；③ 邊界附近紅藍混著、到處有落單的異類——那是翻掉的那 10 家。

### A3・畫圖工具（給現成）

後面五種都要畫一次，先包成一個工具。

In [ ]:
from sklearn.inspection import DecisionBoundaryDisplay

def plot_boundary(model, title):
    """把模型的判斷畫成底色，再把 100 家公司點上去"""
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    DecisionBoundaryDisplay.from_estimator(
        model, X, ax=ax, cmap=cmap_bg, alpha=1.0,
        response_method="predict", grid_resolution=300,
        xlabel="毛利率 (%)", ylabel="負債比 (%)")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap_pt, edgecolors="k", s=45)
    ax.set_title(title)
    plt.show()

print("畫圖工具準備好了")

**🔑 怎麼看：底色＝模型的判斷，圓點＝真實答案。點色跟底色一樣就是猜對。⭐ 重點看底色的形狀。**

---
## 🟩 B・第一種：KNN（小美的做法）

去舊資料裡找**最像的 k 家**，看那 k 家漲的多還是跌的多，就跟著判。

In [ ]:
# TODO：匯入 KNN 分類器。名字＝「K 個最近鄰居」的英文接上 Classifier
from sklearn.neighbors import ____

# TODO：用剛剛匯入的那個建一個模型，k 設成 5
knn = ____(n_neighbors=5)    # k=5：看最近的 5 家
knn.fit(X, y)                                # KNN 的 fit 沒在算，只是把 100 家記下來

new = [[50, 30]]                             # 一家新公司：毛利率 50%、負債比 30%（體質好）
print("新公司 毛利率50 負債比30 → 預測：", knn.predict(new)[0], "（1＝漲、0＝跌）")
print("考自己的答對率：", round(knn.score(X, y), 3))

> **⚠️ 上面那個數字先別當真**——我們是拿它**已經看過的 100 家**去考它，等於拿念過的考卷考自己。
> **先看圖的形狀，不要比數字。** 為什麼這個數字騙人，**今天最後一段會算給你看**。

In [ ]:
plot_boundary(knn, "KNN k=5")

**🔑 邊界是不規則的曲線，完全跟著點走。**

### B2・把 k 轉到兩個極端

In [ ]:
# TODO：把 k 轉到最小的極端 —— 只看最近的 1 家
knn1 = KNeighborsClassifier(n_neighbors=____)
knn1.fit(X, y)
plot_boundary(knn1, "KNN k=1   考自己 " + str(round(knn1.score(X, y), 3)))

In [ ]:
# TODO：換到另一個極端 —— 看最近的 15 家
knn15 = KNeighborsClassifier(n_neighbors=____)
knn15.fit(X, y)
plot_boundary(knn15, "KNN k=15   考自己 " + str(round(knn15.score(X, y), 3)))

**🔑 ⭐ k=1 考自己 1.000，但藍海裡浮著一堆紅色小島**——每個島就是一家異類。它一定全對，因為離每家最近的就是它自己，**照抄而已**。

**k=15 掉到 0.86，邊界卻變成平順的斜線**，小島全消失。

**➡️ k ＝「要不要理會少數異類」的旋鈕。**

---
## 🟧 C・第二種：決策樹（阿凱的做法）

不比對誰跟誰像，而是**問問題**、一路分岔。`max_depth` ＝ 最多問幾層。

In [ ]:
# TODO：匯入決策樹分類器。名字＝「決策樹」的英文接上 Classifier
from sklearn.tree import ____, export_text

# TODO：跟上一個一樣的寫法，深度上限設 2
tree2 = ____(max_depth=2, random_state=42)   # 最多問 2 層
tree2.fit(X, y)                                                # 決策樹的 fit 是真的在算：自己找該問哪個、門檻多少

print("新公司 毛利率50 負債比30 → 預測：", tree2.predict(new)[0])
print("考自己的答對率：", round(tree2.score(X, y), 3))

### C2・招牌好料一：把整棵樹印成規則

In [ ]:
feature_names = ["margin", "debt"]
print(export_text(tree2, feature_names=feature_names))

**🔑 這幾行就是模型的全部腦袋，門檻是它自己算的。⚠️ 每條規則都是「某個欄位 ≤ 某個數字」——一次只看一個欄位。**

In [ ]:
plot_boundary(tree2, "決策樹 max_depth=2   考自己 " + str(round(tree2.score(X, y), 3)))

**🔑 ⭐ 全部是橫的和豎的，沒有斜線**——因為它一次只切一刀、而且一定平行於某根軸。**真規則是斜線，所以它只能用階梯逼近。**

### C3・把深度放到底

In [ ]:
# TODO：填 Python 裡代表「沒有值」的那個字，意思是不限制深度（⚠️ 不是字串，不用引號）
tree_none = DecisionTreeClassifier(max_depth=____, random_state=42)   # 不限深度，讓它一直問下去
tree_none.fit(X, y)
plot_boundary(tree_none, "決策樹 不限深度   考自己 " + str(round(tree_none.score(X, y), 3)))

**🔑 ⭐⭐ 又是 1.000，階梯碎成一堆細長條**——每條都是為了框住某一家異類多開的分岔。

**跟 k=1 是同一件事，換一種模型演。**

### C4・招牌好料二：哪個欄位比較重要

In [ ]:
# TODO：決策樹的招牌屬性 —— 每個欄位有多重要。⚠️ 結尾有一個底線
importances = tree2.____
for i in range(len(feature_names)):
    print(feature_names[i], "重要性：", round(importances[i], 3))

**🔑 兩個加起來是 1.000，誰大誰就是樹主要拿來分岔的。KNN 沒有這個東西。**

---
## 🟨 D・第三種：隨機森林（全店店員一起投票）

種很多棵樹，每棵只看**一部分資料、一部分欄位**，最後**全部投票**。`n_estimators` ＝ 種幾棵。

In [ ]:
# TODO：匯入隨機森林分類器。名字＝「隨機森林」的英文接上 Classifier
from sklearn.ensemble import ____

# TODO：第三種，寫法跟前兩種一模一樣，種 200 棵
forest = ____(n_estimators=200, random_state=42)   # 種 200 棵
forest.fit(X, y)

print("新公司 毛利率50 負債比30 → 預測：", forest.predict(new)[0])
print("考自己的答對率：", round(forest.score(X, y), 3))

In [ ]:
plot_boundary(forest, "隨機森林 200 棵   考自己 " + str(round(forest.score(X, y), 3)))

**🔑 還是階梯（裡面每棵都是決策樹），但又細又密、貼著斜線。**

**⚠️ 考自己也是 1.000，跟那棵背死答案的樹一樣**——光看數字分不出好壞。**先用眼睛看：它的邊界平順多了。**

### D2・把種樹的過程畫出來

In [ ]:
counts = [1, 5, 200]

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for i in range(len(counts)):
    n = counts[i]
    m = RandomForestClassifier(n_estimators=n, random_state=42)
    m.fit(X, y)
    ax = axes[i]
    DecisionBoundaryDisplay.from_estimator(
        m, X, ax=ax, cmap=cmap_bg, alpha=1.0,
        response_method="predict", grid_resolution=200,
        xlabel="毛利率 (%)", ylabel="負債比 (%)")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap_pt, edgecolors="k", s=35)
    ax.set_title("隨機森林 " + str(n) + " 棵   考自己 " + str(round(m.score(X, y), 3)))
plt.show()

**🔑 ⭐ 破碎的小塊被一票一票平均掉了。** 每棵樹被**不同的**異類騙到，200 棵投票時互相蓋過去。

In [ ]:
importances = forest.feature_importances_
for i in range(len(feature_names)):
    print(feature_names[i], "重要性：", round(importances[i], 3))

**🔑 森林的重要性是每棵樹平均起來的，比單棵樹的更可信。**

---
## 🟪 E・第四種：XGBoost（阿凱一版一版改清單）

先種一棵很淺很粗的樹，**看它哪裡錯，第二棵專門補那些錯**，接力幾百棵。業界表格型資料最常用的之一。

In [ ]:
# TODO：匯入 XGBoost 的分類器。名字＝XGB 接上 Classifier
from xgboost import ____

# TODO：第四種，還是同樣的寫法
xgb = ____(n_estimators=200, max_depth=3, learning_rate=0.3,
                    random_state=42, eval_metric="logloss")
xgb.fit(X, y)

print("新公司 毛利率50 負債比30 → 預測：", xgb.predict(new)[0])
print("考自己的答對率：", round(xgb.score(X, y), 3))

In [ ]:
plot_boundary(xgb, "XGBoost 200 棵   考自己 " + str(round(xgb.score(X, y), 3)))

**🔑 跟隨機森林很像——光看成品分不出差別。差別在「怎麼做出來的」，下一格畫給你看。**

### E2・把接力的過程畫出來，跟上面那排比

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for i in range(len(counts)):
    n = counts[i]
    m = XGBClassifier(n_estimators=n, max_depth=3, learning_rate=0.3,
                      random_state=42, eval_metric="logloss")
    m.fit(X, y)
    ax = axes[i]
    DecisionBoundaryDisplay.from_estimator(
        m, X, ax=ax, cmap=cmap_bg, alpha=1.0,
        response_method="predict", grid_resolution=200,
        xlabel="毛利率 (%)", ylabel="負債比 (%)")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap_pt, edgecolors="k", s=35)
    ax.set_title("XGBoost " + str(n) + " 棵   考自己 " + str(round(m.score(X, y), 3)))
plt.show()

**🔑 ⭐ 跟上面那排疊起來看：**

| | 第 1 棵 | 加棵數在做什麼 |
|---|---|---|
| 隨機森林 | 已經很破碎 | **把破碎平均掉** → 越來越平滑 |
| XGBoost | 一個粗方塊 | **一階階補細節** → 越來越貼合 |

**森林＝一群專家各自作答再投票；XGBoost＝一個人一直改考卷。**

---
## 🟥 F・把考卷藏起來，重考一次

四種分類器都跑過了，但有件事說不通：**KNN k=1、決策樹不限深、隨機森林——考自己都是 1.000，畫出來的邊界卻天差地遠。**

問題出在**我們一直拿它已經看過的 100 家去考它**。這一段換個考法。

### F1・先認四個名詞

| 詞 | 白話 |
|---|---|
| **樣本內**（train） | 給模型念的那份，就是前面一直說的「考自己」 |
| **樣本外**（test） | **藏起來**、最後才拿出來考的那份 |
| **過擬合** | 樣本內超高、樣本外很差 —— **把答案背死了** |
| **欠擬合** | 兩邊都低 —— **根本沒學會** |

### F2・切一刀：70 家念書、30 家當考卷

`train_test_split` 會**隨機**抽出 30 家藏起來。公司之間沒有先後順序，隨機抽沒問題。

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=30, random_state=42, stratify=y)   # 隨機抽 30 家藏起來當考卷

print("樣本內（念書）：", len(X_train), "家")
print("樣本外（考卷）：", len(X_test), "家")

### F3・⭐ 四種一起重考

**前面那個說不通的地方，就在這一格解開。**

In [ ]:
models = [
    ("KNN k=1      ", KNeighborsClassifier(n_neighbors=1)),
    ("KNN k=5      ", KNeighborsClassifier(n_neighbors=5)),
    ("KNN k=15     ", KNeighborsClassifier(n_neighbors=15)),
    ("決策樹 深度2  ", DecisionTreeClassifier(max_depth=2, random_state=42)),
    ("決策樹 不限深 ", DecisionTreeClassifier(max_depth=None, random_state=42)),
    ("隨機森林      ", RandomForestClassifier(n_estimators=200, random_state=42)),
    ("XGBoost      ", XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.3,
                                    random_state=42, eval_metric="logloss")),
]

for name, model in models:
    model.fit(X_train, y_train)
    in_score = model.score(X_train, y_train)
    out_score = model.score(X_test, y_test)
    print(name, "樣本內", round(in_score, 3), " 樣本外", round(out_score, 3),
          " 差距", round(in_score - out_score, 3))

**🔑 ⭐ 解開了：三種樣本內都是 1.000，但樣本外——k=1 和不限深的樹掉到 0.800，隨機森林 0.867 最高。**

**森林的 1.000 跟另外兩種的 1.000 不一樣**，樣本內看不出來，樣本外一考就分出來了。

**🔑 決策樹深度 2 兩邊都低（0.786 / 0.733）＝ 欠擬合。**

### F4・把那 30 家考卷畫回圖上

▲ 三角形 ＝ 藏起來的 30 家。**看模型在三角形上錯了哪些。**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
show = [("KNN k=1", models[0][1]), ("隨機森林", models[5][1])]

for i in range(len(show)):
    name, model = show[i]
    ax = axes[i]
    DecisionBoundaryDisplay.from_estimator(
        model, X, ax=ax, cmap=cmap_bg, alpha=1.0,
        response_method="predict", grid_resolution=300,
        xlabel="毛利率 (%)", ylabel="負債比 (%)")
    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap=cmap_pt, s=30, alpha=0.35)
    ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap=cmap_pt,
               s=110, marker="^", edgecolors="k", linewidths=1.5)
    ax.set_title(name + "   樣本外 " + str(round(model.score(X_test, y_test), 3)))
plt.show()

**🔑 左邊 k=1 那些紅色小島，是為了訓練資料裡的異類硬圈出來的**——考卷上的三角形落進去就錯了。右邊森林的邊界乾淨，踩雷的少。

---
## 🟫 G・第五種：KMeans（沒有答案的時候）

前面四種**我們都給了它答案**。但真實工作常常沒有答案——只能問「哪些彼此比較像」。

**⭐ 注意下一格：`fit` 裡面只有 `X`，沒有 `y`。**

In [ ]:
# TODO：匯入 KMeans（它在 sklearn.cluster 底下，cluster ＝分群）
from sklearn.cluster import ____

# TODO：第五種，一樣的寫法 —— 但它是分群，要告訴它分成 2 群
kmeans = ____(n_clusters=2, n_init=10, random_state=42)
kmeans.fit(X)                      # ⭐ 只給 X，完全沒給答案 y

print("每家公司被分到第幾群（前 20 家）：")
print(kmeans.labels_[:20])
print()
print("兩個群中心（毛利率, 負債比）：")
print(kmeans.cluster_centers_.round(1))

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
DecisionBoundaryDisplay.from_estimator(
    kmeans, X, ax=ax, cmap=ListedColormap(["#e8e8e8", "#fff3cc"]), alpha=1.0,
    response_method="predict", grid_resolution=300,
    xlabel="毛利率 (%)", ylabel="負債比 (%)")
ax.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap_pt, edgecolors="k", s=45)
ax.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
           marker="X", s=300, c="black")
ax.set_title("KMeans 分 2 群（底色＝它分的群、圓點＝真實漲跌）")
plt.show()

**🔑 分兩層看：底色＝它分的群（✕ 是群中心），圓點＝真實漲跌（它沒看過）。**

**⭐ 每一塊底色裡面，紅點和藍點都混在一起。**

### G2・它到底分出了什麼？

手上剛好有兩份答案可以對：**漲跌**，還有編資料時分的**族群**（前 50 家一群、後 50 家一群）。

In [ ]:
group = np.array([0] * 50 + [1] * 50)      # 編資料時用的族群：前 50 家一群、後 50 家一群

# 群的編號 0/1 是隨便給的，所以兩種對法都算，取比較高的那個
agree_label = max((kmeans.labels_ == y).mean(), (kmeans.labels_ != y).mean())
agree_group = max((kmeans.labels_ == group).mean(), (kmeans.labels_ != group).mean())

print("KMeans 分的群，對得上「漲跌」的比例：", round(agree_label, 2))
print("KMeans 分的群，對得上「族群」的比例：", round(agree_group, 2))

**🔑 ⭐⭐ 對族群 0.98、對漲跌 0.56。**

**它不是分錯了，是在回答另一個問題**——「哪些公司彼此比較像」，不是「誰會漲」。沒有人告訴它有兩種公司，它自己從數字的疏密看出來的。

**⚠️ 沒有答案的東西，它變不出答案——分群不能拿來預測。**

---
## 🎬 收尾

| 模型 | 給答案 | 邊界長相 | 旋鈕 |
|---|---|---|---|
| KNN | 給 | 不規則曲線 | `n_neighbors` |
| 決策樹 | 給 | **直角階梯** | `max_depth` |
| 隨機森林 | 給 | 細密階梯、平滑 | `n_estimators` |
| XGBoost | 給 | 細密階梯、貼合 | `n_estimators` |
| KMeans | **不給** | 直線切開 | `n_clusters` |

**寫法幾乎一樣：建模型 → `fit` → `predict`。** 只有 KMeans 的 `fit` 不吃答案。

**⭐ 今天最該帶走的一句：看樣本外，別看樣本內。** 樣本內 1.000 可能是真的好，也可能是把答案背死了——**分不出來，就切一刀考它**。

---

### ➡️ 但這份資料，其實是我們自己編的

我們**親手放了一條規則**進去（毛利率減負債比），只加了 10 家例外。所以模型考得出 0.80～0.87，**是因為裡面真的有東西可以學**。

**真實市場也這麼客氣嗎？**

**下半場換上真的台灣股票**，同樣四種模型、同樣切一刀，看看數字會變成什麼樣子。